In [ ]:
!pip install -q ultralytics roboflow

from roboflow import Roboflow
from ultralytics import YOLO
import os, yaml

In [ ]:
from google.colab import userdata

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("cbas-workspace").project("kampus-altyapi-kusur_kaldirim")
version = project.version(6)
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset konumu:", dataset.location)
print("Kullanılan data.yaml:", DATA_YAML)

In [ ]:
model = YOLO("yolov8m.pt")

RUN_NAME = "kaldirim_tespit"

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=768,
    batch=32,
    device=0,
    workers=8,
    patience=15,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    #roboflowda bu işlemler yapıldığı için burada tekrar yapmıyoruz.
    mosaic=0,
    mixup=0,
    copy_paste=0.0,
    close_mosaic=0,
    degrees=0,
    translate=0,
    scale=0,
    shear=0.0,
    perspective=0.0,
    fliplr=0,
    flipud=0.0,
    hsv_h=0,
    hsv_s=0,
    hsv_v=0,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    seed=42,
    amp=True,
    cache="ram",

    project="/content/runs/detect",
    name=RUN_NAME,
    exist_ok=True
)

print("\nEğitim tamamlandı!")
print("Model:", f"/content/runs/detect/{RUN_NAME}/weights/best.pt")

In [ ]:
# Sonuçları değerlendir
model_best = YOLO(f"/content/runs/detect/{RUN_NAME}/weights/best.pt")
metrics = model_best.val(data=DATA_YAML)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f"/content/{RUN_NAME}", "zip", f"/content/runs/detect/{RUN_NAME}")
files.download(f"/content/{RUN_NAME}.zip")